# Historical Transform — Bronze → Silver
Reads `bronze/historical/air_quality_historical.csv` from GCS,
cleans it, and writes Parquet to `silver/historical/`.

In [1]:
import os
os.environ.pop("SPARK_HOME", None)
os.environ.pop("PYSPARK_PYTHON", None)
# os.environ["PYSPARK_PYTHON"] = r"D:\data-engineering-zoomcamp\project\.venv\Scripts\python.exe"
# os.environ["PYSPARK_DRIVER_PYTHON"] = r"D:\data-engineering-zoomcamp\project\.venv\Scripts\python.exe"

# NOTE: PYSPARK_PYTHON is set in ~/.bashrc. Uncomment above if Spark picks up wrong Python.

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

BUCKET = 'hcm-air-quality-486008'
SA_KEY = os.path.abspath('../keys/hcm-pipeline-sa.json')

print('SA key exists:', os.path.exists(SA_KEY))

SA key exists: True


In [3]:
# Building SparkSession with GCS connector
# First run: downloads the GCS connector jar from Maven (~40MB) — takes a minute
# After that it's cached, so subsequent runs are fast
spark = SparkSession.builder \
    .appName('hcm-historical') \
    .config('spark.jars.packages', 'com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.22') \
    .config('spark.hadoop.google.cloud.auth.service.account.enable', 'true') \
    .config('spark.hadoop.google.cloud.auth.service.account.json.keyfile', SA_KEY) \
    .config('spark.hadoop.fs.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem') \
    .config('spark.hadoop.fs.AbstractFileSystem.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

Spark version: 4.1.1


In [5]:
spark

In [6]:
# Read the raw CSV from GCS

df = spark.read \
    .option('header', 'true') \
    .csv(f'gs://{BUCKET}/bronze/historical/air_quality_historical.csv')

print('Row count:', df.count())

Row count: 1298


In [7]:
df.limit(5).toPandas()

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,01-08-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02-08-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03-08-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04-08-22,32.81764706,22.68235294,464.8235294,36.63529412,14.00588235,55.52941176,0.286470588,0,2.182352941,NaN,NaN
4,05-08-22,26.89166667,18.57083333,460.2916667,33.3375,10.15416667,30.375,0.15125,0,1.8375,67.11764706,41.35294118


In [10]:
# Check the schema — everything will be string because we didn't use inferSchema
# We'll cast manually in the next cells
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- pm10: string (nullable = true)
 |-- pm2_5: string (nullable = true)
 |-- carbon_monoxide: string (nullable = true)
 |-- nitrogen_dioxide: string (nullable = true)
 |-- sulphur_dioxide: string (nullable = true)
 |-- ozone: string (nullable = true)
 |-- aerosol_optical_depth: string (nullable = true)
 |-- dust: string (nullable = true)
 |-- uv_index: string (nullable = true)
 |-- us_aqi: string (nullable = true)
 |-- european_aqi: string (nullable = true)



In [11]:
# Fix the date column
# Raw format is DD-MM-YY (e.g. 15-08-22)
# We want YYYY-MM-DD (e.g. 2022-08-15)

df = df.withColumn(
    'date',
    F.date_format(F.to_date(F.col('date'), 'dd-MM-yy'), 'yyyy-MM-dd')
)

# Check: first few dates should look like 2022-08-xx
df.select('date').show(5)

+----------+
|      date|
+----------+
|2022-08-04|
|2022-08-05|
|2022-08-06|
|2022-08-07|
|2022-08-08|
+----------+
only showing top 5 rows


In [13]:
# Casr all pollutant columns from string to double 
numeric_cols = [
    'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide',
    'sulphur_dioxide', 'ozone', 'aerosol_optical_depth',
    'dust', 'uv_index', 'us_aqi', 'european_aqi',
]

for col in numeric_cols:
    df = df.withColumn(col, F.col(col).cast('double'))

# Check schema again - all pollutants should now be double
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm2_5: double (nullable = true)
 |-- carbon_monoxide: double (nullable = true)
 |-- nitrogen_dioxide: double (nullable = true)
 |-- sulphur_dioxide: double (nullable = true)
 |-- ozone: double (nullable = true)
 |-- aerosol_optical_depth: double (nullable = true)
 |-- dust: double (nullable = true)
 |-- uv_index: double (nullable = true)
 |-- us_aqi: double (nullable = true)
 |-- european_aqi: double (nullable = true)



In [17]:
# Check for nulls - how many rows have ALL pollutants null?
from pyspark.sql.functions import col, when

all_null_count = df.filter(
    F.coalesce(*[F.col(c) for c in numeric_cols]).isNull()
).count()

print('Rows where every pollutant is null:', all_null_count)

Rows where every pollutant is null: 0


In [19]:
# Drop rows where every pollutant is null (keep partial nulls)
df = df.dropna(how='all', subset=numeric_cols)

print('Row count after dropping all-null rows:', df.count())

Row count after dropping all-null rows: 1295


In [22]:
# Add ingested_at column so we know when this was processed 

df = df.withColumn('ingested_at', F.current_timestamp())

df.limit(5).toPandas()

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi,ingested_at
0,2022-08-04,32.817647,22.682353,464.823529,36.635294,14.005882,55.529412,0.286471,0.0,2.182353,NaN,NaN,2026-04-09 20:58:26.083977
1,2022-08-05,26.891667,18.570833,460.291667,33.337500,10.154167,30.375000,0.151250,0.0,1.837500,67.117647,41.352941,2026-04-09 20:58:26.083977
2,2022-08-06,27.033333,18.629167,443.583333,34.637500,10.062500,21.125000,0.230417,0.0,1.552083,63.166667,36.375000,2026-04-09 20:58:26.083977
3,2022-08-07,29.616667,20.250000,470.000000,34.654167,12.400000,21.458333,0.235000,0.0,1.333333,66.375000,39.500000,2026-04-09 20:58:26.083977
4,2022-08-08,35.541667,24.429167,545.625000,43.975000,13.150000,21.541667,0.185000,0.0,0.666667,72.416667,50.166667,2026-04-09 20:58:26.083977


In [23]:
# Write clean data to silver layer as Parquet

dest = f'gs://{BUCKET}/silver/historical/'

df.write.mode('overwrite').parquet(dest)

print('✅ Done —>', dest)

✅ Done —> gs://hcm-air-quality-486008/silver/historical/


In [25]:
spark.stop()